In [36]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"

In [37]:
v = embed.encode(q)
v[0]

np.float64(-0.02058203437252893)

In [38]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
	repo_owner="DataTalksClub",
	repo_name="llm-zoomcamp",
    	commit_id="8c1834d",
    	allowed_extensions={"md"},
    	filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [39]:
documents[10]

{'content': "# Agents\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn Part 1 of this module we built RAG pipelines.\n\nEvery pipeline we wrote followed the same flow:\n\n- search the FAQ,\n- build a prompt with the results,\n- send it to the LLM.\n\nThis returns good answers when the user's query matches the documents.\nThe search finds the right entry, the LLM reads it, and you get a\nhelpful reply.\n\nOften, though, the search returns nothing useful.\n\n- Maybe the user made a typo.\n- Maybe they asked the question in an unusual way.\n- Maybe they need information from two different searches.\n\nWe use lexical search here, so the search looks for an exact match.\nOne typo and it misses the entry it needed. In our pipeline there's\nno recovery. The search runs once, and if it returns garbage the LLM\ngets garbage. Our pipeline always does the same thing, no matter what.\n\nInstead of routing the user question str

In [40]:
doc = None

for d in documents:
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        doc = d
        break
    
v_doc = embed.encode(doc["content"])

score = v.dot(v_doc)

score

np.float64(0.36107027225589694)

In [42]:
from gitsource import chunk_documents
from tqdm.auto import tqdm
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

texts = [chunk["content"] for chunk in chunks]

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
	batch = texts[i:i+batch_size]
	batch_vectors = embed.encode_batch(batch)
	X.extend(batch_vectors)

X = np.array(X)

scores = X.dot(v)
idx = np.argmax(scores)
print(chunks[idx]["filename"])

  0%|          | 0/6 [00:00<?, ?it/s]

02-vector-search/lessons/07-sqlitesearch-vector.md


In [46]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(X, chunks)

q4 = 'What metric do we use to evaluate a search engine?'
v4 = embed.encode(q4)
result = vindex.search(v4, num_results=1)
print(result[0]['filename'])

04-evaluation/lessons/05-search-metrics.md


In [51]:
from minsearch import Index

q5 = 'How do I store vectors in PostgreSQL?'
v5 = embed.encode(q5)

text_index = Index(text_fields=['content'], keyword_fields=['filename'])
text_index.fit(chunks)

text_result = text_index.search(q5, num_results=5)
vector_result = vindex.search(v5, num_results=5) 

print("-----Text search results-----")
for result in text_result:
    print(result['filename'])
print("-----Vector search results-----")
for result in vector_result:
    print(result['filename'])


-----Text search results-----
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
-----Vector search results-----
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [52]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [54]:
q6 = 'How do I give the model access to tools?'
v6 = embed.encode(q6)

text_index = Index(text_fields=['content'], keyword_fields=['filename'])
text_index.fit(chunks)

text_result = text_index.search(q6, num_results=5)
vector_result = vindex.search(v6, num_results=5) 

hybrid_results = rrf([text_result, vector_result], k=60, num_results=5)

print(hybrid_results[0]['filename'])

01-agentic-rag/lessons/13-function-calling.md
